# Qwen3.5-0.8B × TinyCeNN — Integrated Memory V2

V2 fixes cached decoding when the **first Qwen3.5 full-attention layer (layer 3)** is replaced by TinyCeNN.

Qwen3.5-0.8B text backbone: 24 layers = 18 native Gated DeltaNet `linear_attention` layers + 6 `full_attention` layers at **3, 7, 11, 15, 19, 23**.

- conservative CeNN: replace `3,23`
- expanded CeNN: replace all six full-attention layers
- validation NLL chooses only between CeNN candidates
- held-out test data is never used for selection
- final cell packages + uploads the selected adapter/model card to Hugging Face


## Setup — always use latest repository code


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

REPO = Path(tempfile.mkdtemp(prefix='qwen35-cenn-v2-')) / 'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)

subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers==5.17.0','huggingface_hub>=0.36.2','datasets>=3,<5',
                'pytest','pandas','matplotlib'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)

os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO),str(REPO/'src')])
sys.path[:0] = [str(REPO),str(REPO/'src')]

import torch
from transformers import AutoConfig
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
cfg = AutoConfig.from_pretrained(MODEL_ID).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == 'full_attention']
LINEAR = [i for i,t in enumerate(cfg.layer_types) if t == 'linear_attention']
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source:', SOURCE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Architecture:', cfg.model_type)
print('Full attention:', FULL)
print('Linear attention:', len(LINEAR), 'layers')
assert cfg.model_type == 'qwen3_5_text'
assert FULL == [3,7,11,15,19,23]


## Configuration


In [ ]:
PROFILE = 'balanced' # @param ['smoke','balanced','extended']
SAVE_TO_DRIVE = True # @param {type:'boolean'}

PROFILES = {
  'smoke': dict(train_contexts='64,128',test_contexts='64,128,256',block_size=16,features=32,train_documents=4,validation_documents=2,test_documents=2,warm_documents=2,warm_steps=2,joint_steps=4,eval_every=2,timing_documents=1,timing_repeats=1,decode_tokens=8,loss_chunk=4),
  'balanced': dict(train_contexts='128,256,512',test_contexts='128,256,512,1024,2048',block_size=32,features=64,train_documents=48,validation_documents=8,test_documents=16,warm_documents=6,warm_steps=40,joint_steps=120,eval_every=20,timing_documents=2,timing_repeats=2,decode_tokens=24,loss_chunk=8),
  'extended': dict(train_contexts='256,512,1024',test_contexts='256,512,1024,2048,4096',block_size=32,features=96,train_documents=96,validation_documents=16,test_documents=32,warm_documents=12,warm_steps=80,joint_steps=240,eval_every=40,timing_documents=3,timing_repeats=3,decode_tokens=32,loss_chunk=8),
}
if not torch.cuda.is_available() and PROFILE != 'smoke':
    raise RuntimeError('Select a GPU runtime or use smoke.')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/TinyCeNN/qwen35-integrated-v2')
else:
    BASE = Path('/content/qwen35-integrated-v2')
BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = BASE / run_id
LOG = BASE / (run_id + '.log')
RUN = dict(PROFILES[PROFILE], seed=2030)
print(json.dumps(RUN, indent=2))
print('Results:', OUT)


## Preflight — includes the layer-3 cache-position regression

This test fails if replacing Qwen's first full-attention layer leaves `DynamicCache.get_seq_length()` at zero during incremental decoding.


In [ ]:
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Testing source:', SOURCE)
env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
r = subprocess.run([sys.executable,'-m','pytest','-q','tests/test_qwen35_integrated_memory.py'],
                   cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f'Qwen3.5 preflight failed: {r.returncode}')
print('✅ Qwen3.5 V2 preflight passed — cache length advances correctly')


## Train + evaluate


In [ ]:
BENCHMARK = REPO / 'scripts/benchmark_qwen35_integrated_memory.py'
cmd = [sys.executable,'-u',str(BENCHMARK),'--base-model',MODEL_ID,'--output-dir',str(OUT)]
for k,v in RUN.items():
    cmd += ['--' + k.replace('_','-'), str(v)]
print('Running:', BENCHMARK.name)
print(' '.join(cmd))
try:
    with LOG.open('w') as log:
        with subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
            status = p.wait()
    if status:
        raise RuntimeError(f'Run failed: {status}; inspect {LOG}')
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG,OUT/'console.log')
        print('Archive:',shutil.make_archive(str(OUT)+'-results','zip',root_dir=OUT))


## Results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
s = pd.read_csv(OUT/'integrated_summary.csv')
selected = json.loads((OUT/'selection.json').read_text())['selected']
cols = [c for c in ['candidate','context','test_nll','test_perplexity','ppl_ratio','adapted_ppl_ratio','total_cache_ratio','prefill_speedup','decode_speedup','remaining_full_attention_layers','cached_logits_nmse','teacher_cached_logits_nmse','candidate_top1_mismatches','teacher_top1_mismatches','selected_on_validation'] if c in s.columns]
print('Locked validation selection:', selected)
display(s[cols].sort_values(['candidate','context']).reset_index(drop=True))
x = s[s.candidate == selected].sort_values('context')
fig,ax = plt.subplots(figsize=(9,4))
ax.plot(x.context,x.ppl_ratio,marker='o',label='PPL ratio')
ax.plot(x.context,x.total_cache_ratio,marker='s',label='cache ratio')
ax.axhline(1,linestyle='--')
ax.set_xscale('log',base=2); ax.set_xlabel('Context'); ax.legend(); ax.set_title(selected); plt.show()


## Publish selected model + generated model card to Hugging Face


In [ ]:
HF_REPO = 'vtava/Qwen3.5-0.8B-CeNN-Integrated-V1'
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Add a Colab secret named HF_TOKEN with write permission.')
pub = [sys.executable,'-u',str(REPO/'scripts/package_qwen35_cenn_hf.py'),
       '--run-dir',str(OUT),'--repo-id',HF_REPO,'--token',HF_TOKEN]
subprocess.run(pub,cwd=REPO,check=True)
print('✅ Published: https://huggingface.co/' + HF_REPO)
